# B1.12 · Severity calibration and reporting

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

Builds on **[B1.11 · Remediation engineering](https://spbreed.github.io/cyber-commons/lessons/B1.11.html)**.

| | |
|---|---|
| Open-source tooling | OpenGrep |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**Stage 15 — Severity calibration and reporting.** The pipeline's output, and
the stage where its credibility is won or lost.

Most severity is a label copied from the rule that fired: this is a CWE-89, so
it is high. That number predicts nothing, because it ignores everything the
pipeline has just learned:

- did stage 12 **confirm it by execution**?
- is it **reachable** from an untrusted entry point (stage 10)?
- what does it **chain into** (stage 13)?
- does it sit in a **historical risk zone** (stage 1)?

Calibrated severity uses all four. A confirmed, reachable finding that chains
into account takeover is not the same as an unvalidated finding in dead code,
even when both are CWE-89.

The second half of this stage is the report, and the useful report is not a
finding count. It is **per-stage economics**: where bugs are caught, where they
escape, and what each escape costs — because that is what decides next
quarter's budget.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 2 · Stage 15 — calibrate severity from what the pipeline learned

In [ ]:
from dataclasses import dataclass

@dataclass
class Finding:
    fid: str; cwe: str; rule_severity: str
    confirmed: bool; reachable: str        # reachable | unknown | unreachable
    chains_into: str                       # "" if none
    historical_risk: float

RULE_SEV = {"low":1,"medium":2,"high":3,"critical":4}
INV = {v:k for k,v in RULE_SEV.items()}
CHAIN_SEV = {"": 0, "data_exposure": 3, "account_takeover": 4, "admin_actions": 4}

FINDINGS = [
 Finding("F-01","CWE-89","high",  True,  "reachable",   "account_takeover", 0.82),
 Finding("F-02","CWE-89","high",  False, "unreachable", "", 0.10),
 Finding("F-03","CWE-22","medium",True,  "reachable",   "data_exposure", 0.91),
 Finding("F-04","CWE-798","high", False, "unknown",     "", 0.05),
 Finding("F-05","CWE-352","medium",True, "reachable",   "account_takeover", 0.30),
 Finding("F-06","CWE-89","high",  False, "unknown",     "", 0.40),
]

def calibrate(f):
    base = RULE_SEV[f.rule_severity]
    score = base
    why = [f"rule severity {f.rule_severity} ({base})"]
    if f.confirmed:            score += 2; why.append("confirmed by execution (+2)")
    else:                      score -= 1; why.append("not confirmed (-1)")
    if f.reachable == "reachable":     score += 1; why.append("reachable from an entry point (+1)")
    elif f.reachable == "unreachable": score -= 2; why.append("unreachable (-2)")
    else:                              why.append("reachability unknown (0)")
    if f.chains_into:
        score = max(score, CHAIN_SEV[f.chains_into] + 2)
        why.append(f"chains into {f.chains_into} (floor raised)")
    if f.historical_risk > 0.6: score += 1; why.append("in a historical risk zone (+1)")
    band = ("critical" if score >= 6 else "high" if score >= 4
            else "medium" if score >= 2 else "low")
    return {"fid": f.fid, "rule": f.rule_severity, "calibrated": band,
            "score": score, "why": why}

rows = [calibrate(f) for f in FINDINGS]
print(f"{'id':7s}{'rule sev':10s}{'calibrated':12s}{'score':>6}")
print("-" * 40)
for r in rows:
    moved = "" if r["rule"] == r["calibrated"] else "   ← moved"
    print(f"{r['fid']:7s}{r['rule']:10s}{r['calibrated']:12s}{r['score']:>6}{moved}")

## 3 · Where it breaks — rule severity as the queue order

In [ ]:
by_rule = sorted(FINDINGS, key=lambda f: -RULE_SEV[f.rule_severity])
by_cal  = sorted(rows, key=lambda r: -r["score"])

print(f"{'rank':6s}{'by rule severity':22s}{'by calibrated severity':24s}")
print("-" * 56)
for i, (a, b) in enumerate(zip(by_rule, by_cal), 1):
    print(f"{i:<6}{a.fid + ' (' + a.rule_severity + ')':22s}"
          f"{b['fid'] + ' (' + b['calibrated'] + ')':24s}")

top_rule = {f.fid for f in by_rule[:3]}
top_cal  = {r["fid"] for r in by_cal[:3]}
print(f"\ntop-3 by rule       : {sorted(top_rule)}")
print(f"top-3 by calibration: {sorted(top_cal)}")
print(f"disagreement        : {sorted(top_rule ^ top_cal)}")
for r in rows:
    f = next(x for x in FINDINGS if x.fid == r["fid"])
    if r["fid"] in top_rule - top_cal:
        print(f"\n{r['fid']} is high by rule and {r['calibrated']} calibrated because:")
        for w in r["why"]: print(f"   · {w}")
assert top_rule != top_cal

## 4 · The report — per-stage economics, not a finding count

In [ ]:
from dataclasses import dataclass as dc

@dc
class Stage:
    name: str; found: int; escaped: int; false_positives: int; minutes: float

PIPELINE_STAGES = [
 Stage("design",   2,  9,  1,  40),
 Stage("code",    14,  6,  9,  70),
 Stage("review",   9,  4, 22, 110),
 Stage("test",     4,  2,  3,  50),
 Stage("deploy",   1,  1,  1,  20),
 Stage("runtime",  1,  0,  0, 180),
]
ESCAPE_MULTIPLIER = 6.0

print(f"{'stage':9s}{'found':>6}{'escaped':>9}{'FP':>5}{'precision':>11}{'min/find':>10}")
print("-" * 51)
for s in PIPELINE_STAGES:
    total = s.found + s.false_positives
    prec = s.found/total if total else 0
    per = s.minutes/s.found if s.found else 0
    print(f"{s.name:9s}{s.found:>6}{s.escaped:>9}{s.false_positives:>5}{prec:>11.2f}{per:>10.1f}")

def escape_cost(stages, m=ESCAPE_MULTIPLIER):
    n = len(stages)
    return {s.name: round(s.escaped * (m ** (n-i-1)) / 1000, 2)
            for i, s in enumerate(stages)}

costs = escape_cost(PIPELINE_STAGES)
print(f"\n{'stage':9s}{'escaped':>9}{'relative escape cost':>22}")
print("-" * 42)
for s in PIPELINE_STAGES:
    bar = "█" * min(int(costs[s.name] * 2), 34)
    print(f"{s.name:9s}{s.escaped:>9}{costs[s.name]:>14}  {bar}")

In [ ]:
# The one-page report the pipeline actually emits.
def report(findings, calibrated, stages, costs):
    crit = [r for r in calibrated if r["calibrated"] == "critical"]
    confirmed = [f for f in findings if f.confirmed]
    unvalidated = [f for f in findings if not f.confirmed and f.reachable == "unknown"]
    worst_stage = max(costs, key=costs.get)
    return f"""APPSEC PIPELINE REPORT

  findings emitted            {len(findings)}
  confirmed by execution      {len(confirmed)}   (stage 12)
  unvalidated + unknown reach {len(unvalidated)}   ← a gap in probe generation, not a pass
  calibrated critical         {len(crit)}   {[r['fid'] for r in crit]}

  severity is calibrated from: confirmation, reachability, chaining and
  historical risk — not from the rule that fired.

  highest escape cost at stage: {worst_stage} ({costs[worst_stage]})
  → that is where the next analyser should be pointed, not where the
    most findings currently are."""

print(report(FINDINGS, rows, PIPELINE_STAGES, costs))
assert max(costs, key=costs.get) == "design"

## What you just proved

Calibration moves several findings off their rule severity: the confirmed reachable CWE-89 that chains into account takeover becomes critical, while the unreachable and unvalidated ones fall. The top-3 by rule severity and by calibration disagree. The stage table shows review with the worst precision and highest minutes per finding, and design carrying the highest escape cost despite only two findings.

## Your turn

Recalculate severity for your current open findings using confirmation and reachability alone — you do not need chaining to see the effect. The queue reorders, and the items that fall are usually the ones people have been arguing about.

---

**Next → [B1.13 · Context engineering for the pipeline](https://spbreed.github.io/cyber-commons/lessons/B1.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*